# Capstone: Flagging Toxic Comments for Human Moderators

**Author:** Viola Kempa

> **Content warning:** this notebook prints real online comments. Some contain slurs, profanity and threats. They
> appear because the error analysis depends on reading them.

## Executive summary

EXECUTIVE_SUMMARY_PLACEHOLDER

## 1. Setup

The exact package versions are pinned in `requirements.txt`, together with the spaCy English model. The install line
below is left commented out on purpose: uncomment it only in a fresh environment.

In [ ]:
# %pip install -r requirements.txt

Imports and every setting the rest of the notebook reads. `TOXIC_THRESHOLD` turns the annotators' score into a label.
`TRAIN_SIZE` and `VALIDATION_SIZE` control how many comments are sampled from the dataset's own splits; the test split
is used in full. The transformer settings follow the usual DistilBERT fine-tuning recipe. `DEVICE` picks the Apple GPU
(`mps`) when there is one.

In [ ]:
import html
import json
import platform
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import spacy
import torch
import transformers
from captum.attr import LayerIntegratedGradients
from datasets import Dataset, load_dataset
from IPython.display import HTML, display
from scipy.optimize import minimize_scalar
from sklearn.calibration import calibration_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score, f1_score, log_loss,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, PrinterCallback,
                          Trainer, TrainingArguments)

plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.max_colwidth", 200)
transformers.logging.set_verbosity_error()

FIGURES_DIR = Path("figures")
MODEL_DIR = Path("models") / "distilbert_toxicity"
CHECKPOINTS_DIR = Path("models") / "checkpoints"
TRAINING_LOG_PATH = Path("training_log.json")
PREDICTIONS_PATH = Path("test_predictions.parquet")
RESULTS_PATH = Path("results.json")
FIGURES_DIR.mkdir(exist_ok=True)

SEED = 42
DATASET_NAME = "google/civil_comments"
TOXIC_THRESHOLD = 0.5       # share of annotators who rated the comment toxic or very toxic
AMBIGUOUS_BAND = (0.3, 0.7)
TRAIN_SIZE = 50_000
VALIDATION_SIZE = 10_000
EDA_SAMPLE_PER_CLASS = 3_000
CLASS_NAMES = {0: "non-toxic", 1: "toxic"}

TFIDF_C_GRID = [1, 4, 16, 64]

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
EPOCHS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
EVAL_EVERY_STEPS = 500
LOG_EVERY_STEPS = 50

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    transformers.set_seed(seed)


set_seed(SEED)

CONFIG = {
    "seed": SEED, "dataset": DATASET_NAME, "toxic label threshold": TOXIC_THRESHOLD,
    "train comments (sampled)": TRAIN_SIZE, "validation comments (sampled)": VALIDATION_SIZE,
    "test comments": "full test split", "TF-IDF C grid": TFIDF_C_GRID, "transformer": MODEL_NAME,
    "max tokens": MAX_LENGTH, "learning rate": LEARNING_RATE, "batch size": BATCH_SIZE, "epochs": EPOCHS,
    "weight decay": WEIGHT_DECAY, "warmup share of steps": WARMUP_RATIO, "device": DEVICE,
}
VERSIONS = {
    "python": platform.python_version(), "datasets": datasets.__version__, "scikit-learn": sklearn.__version__,
    "spacy": spacy.__version__, "torch": torch.__version__, "transformers": transformers.__version__,
}
results = {"config": CONFIG, "versions": VERSIONS}
display(pd.Series(CONFIG | VERSIONS, dtype=str).to_frame("value"))

## 2. Problem framing

- **Dataset:** Civil Comments (Borkan et al., 2019). It holds about 2 million English comments from around 50
  independent news sites that used the Civil Comments plugin between 2015 and 2017. When the platform shut down, Jigsaw
  (Google) released the comments with crowd-sourced toxicity ratings. Licence CC0, loaded from Hugging Face as
  `google/civil_comments`.
- **Target:** `label = 1` (toxic) when at least half of the annotators rated the comment toxic or very toxic
  (`toxicity >= 0.5`). This is the convention of the Jigsaw competitions built on the same data.
- **Input:** the comment text only. The release has no author, article or date fields.
- **Use case:** a busy comment section receives more comments than its moderators can read. The model sorts the
  moderation queue so that likely toxic comments are read first. A moderator still makes every decision.
- **Stakeholders:** moderators decide which comments to read first, and the site decides the score at which a comment
  jumps the queue. Commenters are affected by both decisions without having a say in either.
- **Out of scope:** removing comments or banning users automatically, other languages, other kinds of platform (social
  media, chat, games) and comments written after 2017.

**Why this dataset.** The text is real and unedited: typos, sarcasm, quotes, shouting and links. The toxic class is
rare (8%), so plain accuracy says almost nothing and the metrics have to be chosen with care. And the dataset was
released because toxicity models had been caught treating identity words as signs of toxicity (Dixon et al., 2018).
That makes the fairness questions in section 10 concrete.

### Hypotheses

- **H1 (model):** fine-tuned DistilBERT reaches a higher test PR-AUC than TF-IDF with logistic regression. Reason: a
  pretrained transformer reads words in context, so it can tell "you are an idiot" from "I felt like an idiot when I
  locked myself out", while TF-IDF only counts words and word pairs.
- **H2 (bias):** both models flag harmless comments that mention an identity group more often than harmless comments
  in general. Reason: identity words show up more often in toxic training comments than in non-toxic ones, and both
  models learn from those co-occurrences.

**Main metric:** PR-AUC (average precision) for the toxic class. It measures how well the model *ranks* toxic comments
above harmless ones, which is the moderation-queue use case, and it is not inflated by the 92% of harmless comments
the way accuracy and ROC-AUC are. Precision, recall and F1 at a fixed threshold are reported next to it.

**Success criteria**, fixed before any test result was seen:

1. DistilBERT reaches an F1 of at least 0.60 on the test set, and beats TF-IDF + logistic regression with a bootstrap
   95% confidence interval of the difference that lies above 0.
2. On every identity slice, the false positive rate is at most twice the overall false positive rate.

## 3. Data

### 3.1 Loading and basic statistics

The Hugging Face release keeps the three splits the dataset authors published. All of them carry the same columns: the
comment `text`, the overall `toxicity` score and six sub-type scores. Each score is the share of crowd annotators who
applied that label, so `toxicity = 0.6` means 6 in 10 annotators called the comment toxic or very toxic.

In [ ]:
civil_comments = load_dataset(DATASET_NAME)


def toxic_share(split):
    return float(np.mean(split.data.column("toxicity").to_numpy() >= TOXIC_THRESHOLD))


display(pd.DataFrame({name: {"comments": f"{len(split):,}", "toxic share": f"{toxic_share(split):.1%}"}
                      for name, split in civil_comments.items()}).T)

### 3.2 Split strategy

The dataset's own split is kept, so no test comment is ever used for training or tuning. Training on all 1.8 million
comments would take days on a laptop GPU, so the notebook draws a random sample of 50,000 training and 10,000
validation comments with a fixed seed. The test split is used in full (97,320 comments). The rare identity groups in
section 10 need every test comment they can get.

In [ ]:
def sample_comments(split_name, size):
    return civil_comments[split_name].shuffle(seed=SEED).select(range(size)).to_pandas()


train_raw = sample_comments("train", TRAIN_SIZE)
validation_raw = sample_comments("validation", VALIDATION_SIZE)
test_raw = civil_comments["test"].to_pandas()
train_raw.sample(5, random_state=SEED)

### 3.3 Cleaning

Cleaning is kept light on purpose. Case, punctuation, stopwords and pronouns all carry information about tone, and the
exploratory analysis in section 4 shows that toxic comments use more second-person pronouns. Two things are fixed:

- Links are replaced with the placeholder `URL`. Their exact address says nothing about toxicity and would only add
  one-off tokens to the vocabulary.
- Line breaks and repeated spaces are collapsed into one space.

Scraped web text often contains HTML entities such as `&amp;`, so the table below also counts those. This release has
none (`&` already appears as a plain character), so no decoding step is needed.

In [ ]:
HTML_ENTITY_PATTERN = re.compile(r"&[a-zA-Z]+;|&#\d+;")
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
WHITESPACE_PATTERN = re.compile(r"\s+")
REPEATED_WHITESPACE_PATTERN = re.compile(r"\n|\s{2,}")


def clean_text(text):
    return WHITESPACE_PATTERN.sub(" ", URL_PATTERN.sub("URL", text)).strip()


def prepare_comments(raw):
    comments = raw.assign(label=(raw["toxicity"] >= TOXIC_THRESHOLD).astype(int),
                          text_clean=raw["text"].map(clean_text))
    return comments[comments["text_clean"] != ""]


CLEANING_RULES = {"HTML entity": HTML_ENTITY_PATTERN, "link": URL_PATTERN,
                  "line break or repeated space": REPEATED_WHITESPACE_PATTERN}
rule_hits = {rule: train_raw["text"].str.contains(pattern) for rule, pattern in CLEANING_RULES.items()}
display(pd.DataFrame({"training comments affected": {rule: f"{hits.sum():,} ({hits.mean():.1%})"
                                                     for rule, hits in rule_hits.items()}}))

def snippet_around_match(text, pattern, width=70):
    match = pattern.search(text)
    return text[max(match.start() - width, 0):match.end() + width]


for rule, hits in rule_hits.items():
    if hits.any():
        snippet = snippet_around_match(train_raw.loc[hits, "text"].iloc[0], CLEANING_RULES[rule])
        print(f"{rule}\n  before: {snippet!r}\n  after:  {clean_text(snippet)!r}\n")

After cleaning, any training or validation comment whose exact text also appears in the test split is removed. Short
stock replies ("Thank you", "Well said") occur many times in the data, and leaving them in both places would let the
models score test comments they have already seen.

In [ ]:
train = prepare_comments(train_raw)
validation = prepare_comments(validation_raw)
test = prepare_comments(test_raw)

test_texts = set(test["text_clean"])
train_in_test = train["text_clean"].isin(test_texts)
validation_in_test = validation["text_clean"].isin(test_texts)
print(f"Removed {train_in_test.sum():,} training and {validation_in_test.sum():,} validation comments "
      f"whose text also appears in the test split")
train = train[~train_in_test].reset_index(drop=True)
validation = validation[~validation_in_test].reset_index(drop=True)

SPLITS = {"train": train, "validation": validation, "test": test}
assert not train["text_clean"].isin(test_texts).any() and not validation["text_clean"].isin(test_texts).any()
assert all(0.05 < split["label"].mean() < 0.12 for split in SPLITS.values()), "unexpected share of toxic comments"
display(pd.DataFrame({name: {"comments": f"{len(split):,}", "toxic": f"{split['label'].sum():,}",
                             "toxic share": f"{split['label'].mean():.1%}"} for name, split in SPLITS.items()}).T)

### 3.4 How sure are the labels?

The label is a vote. A comment that 5 of 10 annotators called toxic becomes `toxic`, and one that 4 of 10 called toxic
becomes `non-toxic`, so labels near the 0.5 cut-off are close to a coin flip.

In [ ]:
fig, ax = plt.subplots()
ax.hist(train["toxicity"], bins=50, color="tab:blue")
ax.axvline(TOXIC_THRESHOLD, color="tab:red", linestyle="--", label=f"label threshold {TOXIC_THRESHOLD}")
ax.axvspan(*AMBIGUOUS_BAND, color="tab:orange", alpha=0.15, label="ambiguous band 0.3 to 0.7")
ax.set_yscale("log")
ax.set_xlabel("toxicity score (share of annotators)")
ax.set_ylabel("training comments (log scale)")
ax.set_title("Most comments are clearly fine; toxic ones are spread out")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "toxicity_score_distribution.png", dpi=150)
plt.show()

in_band = train["toxicity"].between(*AMBIGUOUS_BAND, inclusive="left")
print(f"Score exactly 0: {(train['toxicity'] == 0).mean():.1%} of training comments")
print(f"Score in the ambiguous band: {in_band.mean():.1%} of all comments, "
      f"{in_band[train['label'] == 1].mean():.1%} of toxic comments")

In [ ]:
SUBTYPES = ["severe_toxicity", "obscene", "threat", "insult", "identity_attack", "sexual_explicit"]
toxic_train = train[train["label"] == 1]
display(toxic_train[SUBTYPES].ge(TOXIC_THRESHOLD).mean().map("{:.1%}".format)
        .to_frame("share of toxic training comments with this sub-type"))

## 4. Exploratory text analysis

spaCy's small English pipeline (`en_core_web_sm`) splits 3,000 toxic and 3,000 non-toxic training comments into
sentences and tokens and tags each token with its part of speech (POS). Equal class sizes make the two groups easy to
compare; they do not reflect the real 8% share of toxic comments.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
eda_sample = train.groupby("label").sample(EDA_SAMPLE_PER_CLASS, random_state=SEED).reset_index(drop=True)
eda_docs = list(nlp.pipe(eda_sample["text_clean"], batch_size=200))
eda_names = eda_sample["label"].map(CLASS_NAMES)

### 4.1 Length and structure

In [ ]:
SECOND_PERSON = {"you", "your", "yours", "yourself", "yourselves", "u", "ur"}


def is_word(token):
    return not (token.is_punct or token.is_space)


def describe_structure(doc):
    words = [token for token in doc if is_word(token)]
    word_count = max(len(words), 1)
    return {
        "words": len(words),
        "sentences": sum(1 for _ in doc.sents),
        "all-caps words (%)": 100 * sum(token.is_upper and len(token) > 1 for token in words) / word_count,
        "second-person words (%)": 100 * sum(token.lower_ in SECOND_PERSON for token in words) / word_count,
        "exclamation marks": doc.text.count("!"),
    }


structure = pd.DataFrame([describe_structure(doc) for doc in eda_docs])
display(structure.groupby(eda_names).agg(["median", "mean"]).T.round(2))

### 4.2 Part-of-speech profile

In [ ]:
def pos_shares(docs):
    counts = Counter(token.pos_ for doc in docs for token in doc if not token.is_space)
    return pd.Series(counts) / sum(counts.values())


def docs_of_class(class_name):
    return [doc for doc, name in zip(eda_docs, eda_names) if name == class_name]


pos_profile = pd.DataFrame({name: pos_shares(docs_of_class(name)) for name in CLASS_NAMES.values()}).fillna(0)
pos_profile = pos_profile[pos_profile.min(axis=1) > 0.002].sort_values("non-toxic", ascending=False)
pos_profile["log2 ratio"] = np.log2(pos_profile["toxic"] / pos_profile["non-toxic"])

fig, (ax_share, ax_ratio) = plt.subplots(1, 2, figsize=(13, 4.5))
pos_profile[["non-toxic", "toxic"]].mul(100).plot.bar(ax=ax_share, color=["tab:blue", "tab:red"])
ax_share.set_ylabel("% of tokens")
ax_share.set_title("Share of tokens per part of speech")
ratio = pos_profile["log2 ratio"].sort_values()
ax_ratio.barh(ratio.index, ratio, color=np.where(ratio > 0, "tab:red", "tab:blue"))
ax_ratio.axvline(0, color="black", linewidth=0.8)
ax_ratio.set_xlabel("log2(toxic share / non-toxic share)")
ax_ratio.set_title("Over-represented in toxic comments (right) or non-toxic (left)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pos_profile.png", dpi=150)
plt.show()
display(pos_profile.round(4))

In [ ]:
def top_words(docs, pos, n=15):
    counts = Counter(token.lower_ for doc in docs for token in doc if token.pos_ == pos)
    return [f"{word} ({count})" for word, count in counts.most_common(n)]


display(pd.DataFrame({f"{name}: adjectives": top_words(docs_of_class(name), "ADJ") for name in CLASS_NAMES.values()}
                     | {f"{name}: pronouns": top_words(docs_of_class(name), "PRON") for name in CLASS_NAMES.values()}))

### 4.3 Where the tagger struggles

spaCy's tagger was trained mostly on edited text, and comments are not edited. One visible weak spot is shouting. The
check below looks at adjectives, verbs and adverbs: words that the sample contains in normal case at least five times
and that are usually tagged `ADJ`, `VERB` or `ADV` there. It then counts how often the ALL-CAPS version of the same
word gets a different tag. Nouns are left out because many ALL-CAPS nouns are acronyms ("US", or "GET" for Hawaii's
General Excise Tax) that really are proper nouns.

In [ ]:
CONTENT_TAGS = {"ADJ", "VERB", "ADV"}
TAGGING_EXAMPLE_WORDS = ["EXACTLY", "WORSE", "KNEW"]


def is_shouted(token):
    return token.is_alpha and token.is_upper and len(token) > 2


def is_normal_case(token):
    return token.is_alpha and not token.is_upper


def usual_tags(docs, min_count=5):
    tags = defaultdict(Counter)
    for doc in docs:
        for token in doc:
            if is_normal_case(token):
                tags[token.lower_][token.pos_] += 1
    return {word: counts.most_common(1)[0][0] for word, counts in tags.items() if counts.total() >= min_count}


def tag_disagreements(docs, reference_tags, select):
    rows = []
    for doc in docs:
        for token in doc:
            if select(token) and token.lower_ in reference_tags:
                rows.append({"word": token.text, "tag": token.pos_, "usual tag": reference_tags[token.lower_],
                             "context": token.sent.text[:120]})
    table = pd.DataFrame(rows)
    return table.assign(differs=table["tag"] != table["usual tag"])


reference_tags = {word: tag for word, tag in usual_tags(eda_docs).items() if tag in CONTENT_TAGS}
shouted = tag_disagreements(eda_docs, reference_tags, is_shouted)
normal = tag_disagreements(eda_docs, reference_tags, is_normal_case)
print(f"Tag differs from the word's usual tag: {shouted['differs'].mean():.1%} of {len(shouted):,} ALL-CAPS words, "
      f"{normal['differs'].mean():.1%} of {len(normal):,} normal-case words")
display(shouted[shouted["differs"]]["tag"].value_counts().head(5).to_frame("tag given to ALL-CAPS words that differ"))
display(shouted[shouted["differs"] & shouted["word"].isin(TAGGING_EXAMPLE_WORDS)].drop_duplicates("word"))

### 4.4 Comment length in DistilBERT tokens

The transformer reads at most `MAX_LENGTH` word pieces per comment and cuts off the rest. This plot checks how much text
that cut-off loses.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_token_counts = np.array([len(ids) for ids in tokenizer(train["text_clean"].tolist())["input_ids"]])

fig, ax = plt.subplots()
ax.hist(np.minimum(train_token_counts, 400), bins=80, color="tab:blue")
ax.axvline(MAX_LENGTH, color="tab:red", linestyle="--", label=f"MAX_LENGTH = {MAX_LENGTH}")
ax.set_xlabel("DistilBERT tokens per comment (values above 400 shown at 400)")
ax.set_ylabel("training comments")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "token_lengths.png", dpi=150)
plt.show()
fits = train_token_counts <= MAX_LENGTH
print(f"Median {np.median(train_token_counts):.0f} tokens, "
      f"90th percentile {np.percentile(train_token_counts, 90):.0f}. "
      f"{fits.mean():.1%} of comments fit in {MAX_LENGTH} tokens.")
print(f"Toxic share: {train.loc[fits, 'label'].mean():.1%} of comments that fit, "
      f"{train.loc[~fits, 'label'].mean():.1%} of longer comments")

### 4.5 What this means for the models

- **Length is no shortcut.** Toxic and non-toxic comments have almost the same median length (35 against 34 words)
  and the same median number of sentences (3). A model has to read the content.
- **Toxic comments talk to someone.** They contain more pronouns (log2 ratio +0.15), more second-person words (2.5%
  against 2.0% of words, "you" is the most common pronoun in toxic comments) and almost twice as many exclamation marks
  per comment (0.42 against 0.22). Words like "you" and "your" are on most stopword lists, so **the baseline keeps stopwords**, and word pairs
  (bigrams) let it see "you are" as a unit.
- **Toxic comments react, harmless ones argue.** Interjections are over-represented in toxic comments (+0.37) and
  numbers clearly under-represented (-0.56): harmless comments cite figures more often.
- **Identity words already lean toxic.** "white" and "black" are among the ten most common adjectives in toxic
  comments, next to "stupid" and "ignorant", and neither appears in the non-toxic top fifteen. A model that learns from
  co-occurrence will link these words with toxicity. Section 10 tests whether it does (hypothesis H2).
- **Shouting does not separate the classes but does confuse the tagger.** ALL-CAPS words make up about 1.6% of words
  in both classes, so the uncased DistilBERT loses little by lowercasing. spaCy, however, gives an ALL-CAPS adjective,
  verb or adverb a different tag about 40% of the time, against 12% for the same words in normal case, usually calling
  it a proper noun. POS-based features would be noisy exactly where people shout; neither model uses POS tags.
- **The labels are soft.** 60% of toxic comments sit between 0.5 and 0.7, so a narrow majority of annotators decided
  their label. Some "errors" near the threshold will be disagreements between annotators rather than model mistakes
  (checked in section 8.4).
- **128 tokens are enough for most comments.** 85% of comments fit completely, and longer comments are slightly less
  often toxic (6.7% against 8.1%). Section 8.4 checks whether DistilBERT does worse on the long ones.

## 5. Baseline: TF-IDF and logistic regression

The baseline turns each comment into a TF-IDF vector of single words and word pairs (bigrams). Bigrams keep short
phrases such as "you are" or "shut up" together. `min_df=3` drops terms seen in fewer than three comments, and
`sublinear_tf` dampens a word repeated many times in one comment. Logistic regression with
`class_weight="balanced"` gives the rare toxic comments as much total weight as the non-toxic ones. The `liblinear`
solver is used because it solves this sparse problem to convergence in about a second. In a first attempt, scikit-learn's
default `lbfgs` solver stopped early at its default tolerance, and the validation scores for different `C` values
jumped around by more than the differences between them.

The regularisation strength `C` is picked by PR-AUC on the validation set. The decision threshold is the one that
gives the best F1 on validation. The test set is touched only after both choices are fixed.

In [ ]:
def fit_tfidf_logreg(texts, labels, c):
    model = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True),
        LogisticRegression(C=c, class_weight="balanced", solver="liblinear"),
    )
    return model.fit(texts, labels)


def toxic_proba(model, texts):
    return model.predict_proba(texts)[:, 1]


c_search = {c: fit_tfidf_logreg(train["text_clean"], train["label"], c) for c in TFIDF_C_GRID}
c_scores = {c: average_precision_score(validation["label"], toxic_proba(model, validation["text_clean"]))
            for c, model in c_search.items()}
best_c = max(c_scores, key=c_scores.get)
baseline = c_search[best_c]
display(pd.Series(c_scores).rename_axis("C").to_frame("validation PR-AUC").round(4))
print(f"Chosen C = {best_c}")

Did cleaning help? The same pipeline with the chosen `C`, trained on raw text instead:

In [ ]:
baseline_raw = fit_tfidf_logreg(train["text"], train["label"], best_c)
display(pd.DataFrame({
    "vocabulary size": [len(baseline_raw[0].vocabulary_), len(baseline[0].vocabulary_)],
    "validation PR-AUC": [average_precision_score(validation["label"], toxic_proba(baseline_raw, validation["text"])),
                          c_scores[best_c]],
}, index=["raw text", "cleaned text"]).round(4))

In [ ]:
def best_f1_threshold(labels, probabilities):
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    return float(thresholds[np.argmax(f1[:-1])])


def score_predictions(labels, probabilities, threshold):
    predictions = probabilities >= threshold
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions),
        "F1": f1_score(labels, predictions),
        "ROC-AUC": roc_auc_score(labels, probabilities),
        "PR-AUC": average_precision_score(labels, probabilities),
        "threshold": threshold,
    }


baseline_threshold = best_f1_threshold(validation["label"], toxic_proba(baseline, validation["text_clean"]))
test_probabilities = {"always non-toxic": np.zeros(len(test)),
                      "TF-IDF + LR": toxic_proba(baseline, test["text_clean"])}
thresholds = {"always non-toxic": 0.5, "TF-IDF + LR": baseline_threshold}
baseline_metrics = pd.DataFrame({name: score_predictions(test["label"], probabilities, thresholds[name])
                                 for name, probabilities in test_probabilities.items()}).T
display(baseline_metrics.round(3))

In [ ]:
def plot_confusion_matrix(ax, labels, probabilities, threshold, title):
    ConfusionMatrixDisplay.from_predictions(labels, probabilities >= threshold, ax=ax, colorbar=False,
                                            values_format=",", display_labels=list(CLASS_NAMES.values()))
    ax.set_title(title)


fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion_matrix(ax, test["label"], test_probabilities["TF-IDF + LR"], baseline_threshold,
                      f"TF-IDF + LR, test set, threshold {baseline_threshold:.2f}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix_baseline.png", dpi=150)
plt.show()

Which terms push the baseline towards each class? Logistic regression weights are readable as feature importance, with
one caveat: a weight describes the term in the presence of all the others, and correlated terms share credit.

In [ ]:
vectorizer, classifier = baseline[0], baseline[1]
term_weights = pd.Series(classifier.coef_[0], index=vectorizer.get_feature_names_out()).sort_values()

fig, (ax_clean, ax_toxic) = plt.subplots(1, 2, figsize=(12, 6))
term_weights.head(20).plot.barh(ax=ax_clean, color="tab:blue", title="Strongest pull towards non-toxic")
term_weights.tail(20).plot.barh(ax=ax_toxic, color="tab:red", title="Strongest pull towards toxic")
for ax in (ax_clean, ax_toxic):
    ax.set_xlabel("logistic regression weight")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baseline_term_weights.png", dpi=150)
plt.show()

BASELINE_INTERPRETATION_PLACEHOLDER

## 6. Transformer: fine-tuned DistilBERT

**Model choice.** `distilbert-base-uncased` is a 66-million-parameter version of BERT made by knowledge distillation.
Sanh et al. (2019) report that it is 40% smaller and 60% faster than BERT-base and keeps 97% of its score on the GLUE
language-understanding benchmark. That makes it the strongest model that fine-tunes on 50,000 comments in under an
hour on a laptop GPU.

The *uncased* variant lowercases everything, so it cannot see shouting. Section 4.1 showed that ALL-CAPS words are
about as common in toxic as in non-toxic comments (1.6% against 1.7% of words), so little is lost. The tokenizer splits
words it does not know into known word pieces ("unbelievably" becomes `un`, `##bel`, `##ie`, `##va`, `##bly`), which
helps with typos and invented insults that a TF-IDF vocabulary would simply drop.

**Fine-tuning setup.**

| Setting | Value | Reason |
|---|---|---|
| learning rate | 2e-5 | inside the range Devlin et al. (2019) recommend for BERT fine-tuning (2e-5 to 5e-5) |
| batch size | 32 | one of the two recommended sizes; fits in 16 GB of shared memory |
| epochs | 2 | recommended range is 2 to 4; the checkpoint with the best validation PR-AUC is kept |
| max length | 128 tokens | covers 85% of comments whole (section 4.4) at a quarter of the cost of 512 |
| warmup | first 6% of steps | avoids large updates while the new classification layer is still random |
| weight decay | 0.01 | standard light regularisation for transformer fine-tuning |
| loss | plain cross-entropy | class imbalance is handled by the validation threshold, exactly as for the baseline |
| checkpoint choice | best validation PR-AUC, checked every 500 steps | the same metric the baseline was tuned on |

In [ ]:
def to_token_dataset(comments):
    dataset = Dataset.from_dict({"text": comments["text_clean"].tolist(), "label": comments["label"].tolist()})
    return dataset.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
                       batched=True, remove_columns=["text"])


def toxic_probabilities(logits, temperature=1.0):
    return torch.softmax(torch.as_tensor(logits, dtype=torch.float32) / temperature, dim=-1)[:, 1].numpy()


def validation_pr_auc(evaluation):
    return {"pr_auc": average_precision_score(evaluation.label_ids, toxic_probabilities(evaluation.predictions))}


def training_arguments():
    return TrainingArguments(
        output_dir=str(CHECKPOINTS_DIR), seed=SEED, report_to="none", disable_tqdm=True,
        learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, warmup_steps=WARMUP_RATIO,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=4 * BATCH_SIZE,
        eval_strategy="steps", eval_steps=EVAL_EVERY_STEPS, save_strategy="steps", save_steps=EVAL_EVERY_STEPS,
        logging_steps=LOG_EVERY_STEPS, load_best_model_at_end=True, metric_for_best_model="pr_auc",
        save_total_limit=1, dataloader_pin_memory=False,
    )


def fine_tune_distilbert():
    trainer = Trainer(
        model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2),
        args=training_arguments(), train_dataset=to_token_dataset(train), eval_dataset=to_token_dataset(validation),
        processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=validation_pr_auc,
    )
    trainer.remove_callback(PrinterCallback)
    trainer.train()
    trainer.save_model(MODEL_DIR)
    TRAINING_LOG_PATH.write_text(json.dumps(trainer.state.log_history, indent=2))
    shutil.rmtree(CHECKPOINTS_DIR)

Training takes 40 to 50 minutes on an M1 Pro. It runs only when no saved model exists, so re-running the notebook
reuses the model in `models/distilbert_toxicity/`. Delete that folder to train again from scratch.

In [ ]:
if not MODEL_DIR.exists():
    fine_tune_distilbert()
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()
training_log = json.loads(TRAINING_LOG_PATH.read_text())
print(f"Training time: {training_log[-1]['train_runtime'] / 60:.0f} minutes on {DEVICE}")

In [ ]:
def log_series(log, key):
    return pd.Series({entry["step"]: entry[key] for entry in log if key in entry}, name=key)


validation_curve = log_series(training_log, "eval_pr_auc")
best_step = validation_curve.idxmax()

fig, (ax_loss, ax_score) = plt.subplots(1, 2)
log_series(training_log, "loss").plot(ax=ax_loss, label="training loss", color="tab:blue", alpha=0.7)
log_series(training_log, "eval_loss").plot(ax=ax_loss, label="validation loss", color="tab:red", marker="o")
ax_loss.set_xlabel("training step")
ax_loss.legend()
ax_loss.set_title("Loss")
validation_curve.plot(ax=ax_score, color="tab:red", marker="o")
ax_score.axvline(best_step, color="grey", linestyle="--", label=f"kept checkpoint (step {best_step})")
ax_score.set_xlabel("training step")
ax_score.set_ylabel("validation PR-AUC")
ax_score.legend()
ax_score.set_title("Validation PR-AUC")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves.png", dpi=150)
plt.show()
print(f"Best validation PR-AUC {validation_curve.max():.4f} at step {best_step} of {max(validation_curve.index)}")

TRANSFORMER_TRAINING_NOTE_PLACEHOLDER

## 7. Comparison on the test set

Both models are scored on the same 97,320 test comments. Each one's decision threshold is the value that gave the best
F1 on validation, chosen before the test set was scored. PR-AUC and ROC-AUC do not depend on a threshold.

"Precision at 80% recall" answers the moderator's question: if the queue must contain 4 out of 5 toxic comments, what
share of the flagged comments are really toxic?

In [ ]:
def predict_logits(texts, batch_size=4 * BATCH_SIZE):
    order = np.argsort([len(text) for text in texts])
    batches = []
    for start in range(0, len(texts), batch_size):
        batch_texts = [texts[index] for index in order[start:start + batch_size]]
        encoded = tokenizer(batch_texts, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt")
        with torch.no_grad():
            batches.append(model(**encoded.to(DEVICE)).logits.float().cpu().numpy())
    return np.concatenate(batches)[np.argsort(order)]


def precision_at_recall(labels, probabilities, target_recall=0.8):
    precision, recall, _ = precision_recall_curve(labels, probabilities)
    return float(precision[recall >= target_recall].max())


validation_logits = predict_logits(validation["text_clean"].tolist())
test_logits = predict_logits(test["text_clean"].tolist())
thresholds["DistilBERT"] = best_f1_threshold(validation["label"], toxic_probabilities(validation_logits))
test_probabilities["DistilBERT"] = toxic_probabilities(test_logits)
assert all(len(probabilities) == len(test) for probabilities in test_probabilities.values())

comparison = pd.DataFrame({
    name: score_predictions(test["label"], probabilities, thresholds[name])
    | {"precision at 80% recall": precision_at_recall(test["label"], probabilities)}
    for name, probabilities in test_probabilities.items()
}).T
display(comparison.round(3))

In [ ]:
MODELS = ["TF-IDF + LR", "DistilBERT"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, name in zip(axes, MODELS):
    plot_confusion_matrix(ax, test["label"], test_probabilities[name], thresholds[name],
                          f"{name}, threshold {thresholds[name]:.2f}")
for name, colour in zip(MODELS, ["tab:blue", "tab:red"]):
    precision, recall, _ = precision_recall_curve(test["label"], test_probabilities[name])
    axes[2].plot(recall, precision, color=colour, label=f"{name} (PR-AUC {comparison.loc[name, 'PR-AUC']:.3f})")
    axes[2].scatter(comparison.loc[name, "recall"], comparison.loc[name, "precision"], color=colour, zorder=3)
axes[2].axhline(test["label"].mean(), color="grey", linestyle=":", label="random ranking")
axes[2].set_xlabel("recall")
axes[2].set_ylabel("precision")
axes[2].set_title("Precision-recall curves (dots: chosen thresholds)")
axes[2].legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_comparison.png", dpi=150)
plt.show()

Is the gap bigger than chance? A bootstrap resamples the test set 1,000 times with replacement and recomputes the
difference (DistilBERT minus TF-IDF + LR) each time. The middle 95% of those differences is the confidence interval.

In [ ]:
N_BOOTSTRAP = 1_000


def bootstrap_difference(labels, scores_a, scores_b, metric):
    rng = np.random.default_rng(SEED)
    labels, scores_a, scores_b = (np.asarray(values) for values in (labels, scores_a, scores_b))
    differences = []
    for _ in range(N_BOOTSTRAP):
        rows = rng.integers(0, len(labels), len(labels))
        differences.append(metric(labels[rows], scores_b[rows]) - metric(labels[rows], scores_a[rows]))
    return np.percentile(differences, [2.5, 97.5])


flags = {name: test_probabilities[name] >= thresholds[name] for name in MODELS}
bootstrap = pd.DataFrame({
    "F1": bootstrap_difference(test["label"], flags["TF-IDF + LR"], flags["DistilBERT"], f1_score),
    "PR-AUC": bootstrap_difference(test["label"], test_probabilities["TF-IDF + LR"], test_probabilities["DistilBERT"],
                                   average_precision_score),
}, index=["2.5%", "97.5%"]).T
observed_difference = comparison.loc["DistilBERT", ["F1", "PR-AUC"]] - comparison.loc["TF-IDF + LR", ["F1", "PR-AUC"]]
bootstrap.insert(0, "difference", observed_difference)
display(bootstrap.round(4))

In [ ]:
predictions = test[["text_clean", "toxicity", "label"]].assign(
    tfidf_lr_probability=test_probabilities["TF-IDF + LR"],
    distilbert_probability=test_probabilities["DistilBERT"],
    tfidf_lr_flag=flags["TF-IDF + LR"].astype(int),
    distilbert_flag=flags["DistilBERT"].astype(int),
)
predictions.to_parquet(PREDICTIONS_PATH)
results |= {"baseline C": best_c, "thresholds": thresholds, "test metrics": comparison.to_dict(orient="index"),
            "bootstrap 95% CI of DistilBERT minus TF-IDF + LR": bootstrap.to_dict(orient="index")}
print(f"Saved {len(predictions):,} test predictions to {PREDICTIONS_PATH}")

COMPARISON_INTERPRETATION_PLACEHOLDER

## 8. Error analysis

### 8.1 Curated examples

Twelve test comments: three random hits and three random correct passes, then DistilBERT's three most confident false
positives and three most confident misses. Confident errors are the most useful to read, because they show what the
model is sure about and gets wrong.

In [ ]:
OUTCOME_NAMES = {(1, 1): "TP", (0, 0): "TN", (0, 1): "FP", (1, 0): "FN"}
predictions["outcome"] = [OUTCOME_NAMES[pair] for pair in zip(predictions["label"], predictions["distilbert_flag"])]


def curated_examples(per_outcome=3):
    by_outcome = predictions.groupby("outcome")
    return pd.concat([
        by_outcome.get_group("TP").sample(per_outcome, random_state=SEED),
        by_outcome.get_group("TN").sample(per_outcome, random_state=SEED),
        by_outcome.get_group("FP").nlargest(per_outcome, "distilbert_probability"),
        by_outcome.get_group("FN").nsmallest(per_outcome, "distilbert_probability"),
    ])


EXAMPLE_NOTES = {}

examples = curated_examples()
display(examples[["outcome", "toxicity", "tfidf_lr_probability", "distilbert_probability", "text_clean"]]
        .assign(note=examples.index.map(EXAMPLE_NOTES)).round(3))

### 8.2 Where the two models disagree

In [ ]:
display(pd.crosstab(predictions["tfidf_lr_flag"] == predictions["label"],
                     predictions["distilbert_flag"] == predictions["label"],
                     rownames=["TF-IDF + LR correct"], colnames=["DistilBERT correct"], margins=True))

### 8.3 Fifty errors read and tagged by hand

A random sample of 25 false positives and 25 false negatives from DistilBERT, each read and given the one category that
best explains the mistake. The tags are a judgement call by one reader, and fifty comments give rough shares only.

In [ ]:
ERROR_TAGS = defaultdict(lambda: "untagged")

error_sample = pd.concat([predictions[predictions["outcome"] == outcome].sample(25, random_state=SEED)
                          for outcome in ["FP", "FN"]])
error_sample["category"] = error_sample.index.map(ERROR_TAGS)
assert error_sample["category"].notna().all(), "every sampled error needs a tag"
error_counts = pd.crosstab(error_sample["category"], error_sample["outcome"], margins=True, margins_name="total")
display(error_counts.sort_values("total", ascending=False))

### 8.4 Errors, label certainty and length

Two checks that link the errors back to the data. The first groups test comments by their annotator score: if the
models mostly fail where annotators disagree too, part of the "error" is label noise. The second compares comments that
fit in `MAX_LENGTH` tokens with longer ones that DistilBERT only reads in part.

In [ ]:
SCORE_BINS = [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.01]
SCORE_BIN_LABELS = ["0 to 0.1", "0.1 to 0.3", "0.3 to 0.5", "0.5 to 0.7", "0.7 to 0.9", "0.9 to 1"]


FLAG_COLUMNS = {"TF-IDF + LR": "tfidf_lr_flag", "DistilBERT": "distilbert_flag"}


def error_rate_by(groups):
    rates = {"comments": predictions.groupby(groups, observed=True).size()}
    for name, column in FLAG_COLUMNS.items():
        misclassified = predictions[column] != predictions["label"]
        rates[f"{name} error rate"] = misclassified.groupby(groups, observed=True).mean()
    return pd.DataFrame(rates)


score_bin = pd.cut(predictions["toxicity"], SCORE_BINS, labels=SCORE_BIN_LABELS, right=False).rename("annotator score")
errors_by_score = error_rate_by(score_bin)

fig, ax = plt.subplots()
errors_by_score[[f"{name} error rate" for name in MODELS]].plot.bar(ax=ax, color=["tab:blue", "tab:red"], rot=0)
ax.set_xlabel("annotator toxicity score (label is toxic from 0.5)")
ax.set_ylabel("share of comments misclassified")
ax.set_title("Errors pile up where annotators disagree")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "errors_by_annotator_score.png", dpi=150)
plt.show()
display(errors_by_score.round(3))

test_token_counts = np.array([len(ids) for ids in tokenizer(test["text_clean"].tolist())["input_ids"]])
length_group = pd.Series(np.where(test_token_counts <= MAX_LENGTH, f"fits in {MAX_LENGTH} tokens",
                                  f"longer than {MAX_LENGTH} tokens"), index=predictions.index, name="length")
display(error_rate_by(length_group).round(3))

ERROR_INTERPRETATION_PLACEHOLDER

## 9. Explainability: Integrated Gradients

**Method.** Integrated Gradients (Sundararajan et al., 2017) asks how DistilBERT's toxic score changes as the input
moves step by step from a neutral baseline (the same comment with every word replaced by the `[PAD]` token) to the
real comment. Each word piece's share of that change is its attribution. Red words pushed the comment towards *toxic*,
blue words towards *non-toxic*. Word pieces are summed back into whole words for display.

**What it can and cannot claim.** The attributions describe what moved *this* model's score for *this* comment,
relative to *this* baseline. They are not causal claims about language, they do not show why the model learned a
pattern, and they change if the baseline or the number of steps changes. The convergence delta reports how far the
numerical integration is from adding up exactly; values near 0 are fine. As a sanity check, the top three attributed
words are deleted and the comment is scored again: if the attributions mean anything, that deletion should lower the
score more than deleting three random words.

In [ ]:
IG_STEPS = 50
IG_EXAMPLES_PER_OUTCOME = 2


def toxic_logit(input_ids, attention_mask):
    return model(input_ids=input_ids, attention_mask=attention_mask).logits[:, 1]


integrated_gradients = LayerIntegratedGradients(toxic_logit, model.distilbert.embeddings.word_embeddings)


def merge_word_pieces(tokens, scores):
    words = []
    for token, score in zip(tokens, scores):
        if token.startswith("##") and words:
            words[-1] = (words[-1][0] + token[2:], words[-1][1] + score)
        else:
            words.append((token, score))
    return words


def word_attributions(text):
    encoded = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
    baseline_ids = torch.full_like(encoded["input_ids"], tokenizer.pad_token_id)
    baseline_ids[:, 0], baseline_ids[:, -1] = tokenizer.cls_token_id, tokenizer.sep_token_id
    attributions, delta = integrated_gradients.attribute(
        encoded["input_ids"], baselines=baseline_ids, additional_forward_args=(encoded["attention_mask"],),
        n_steps=IG_STEPS, internal_batch_size=IG_STEPS, return_convergence_delta=True)
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    return merge_word_pieces(tokens[1:-1], attributions.sum(dim=-1)[0, 1:-1].tolist()), abs(delta.item())

In [ ]:
def coloured_words(words):
    largest = max(abs(score) for _, score in words) or 1.0
    spans = []
    for word, score in words:
        rgb = "214, 39, 40" if score > 0 else "31, 119, 180"
        spans.append(f'<span style="background-color: rgba({rgb}, {abs(score) / largest:.2f}); padding: 0 2px">'
                     f"{html.escape(word)}</span>")
    return " ".join(spans)


def text_without(words, positions):
    return " ".join(word for index, (word, _) in enumerate(words) if index not in positions)


def deletion_check(words, rng, k=3, random_draws=10):
    scores = np.array([score for _, score in words])
    variants = [text_without(words, set()), text_without(words, set(np.argsort(scores)[-k:]))]
    variants += [text_without(words, set(rng.choice(len(words), size=min(k, len(words)), replace=False)))
                 for _ in range(random_draws)]
    probabilities = toxic_probabilities(predict_logits(variants))
    return {"score": probabilities[0], "drop, top 3 words removed": probabilities[0] - probabilities[1],
            "drop, 3 random words removed": probabilities[0] - probabilities[2:].mean()}

In [ ]:
examples_with_length = examples.assign(words=examples["text_clean"].str.split().str.len())
explained = pd.concat([group.nsmallest(IG_EXAMPLES_PER_OUTCOME, "words")
                       for _, group in examples_with_length.groupby("outcome", sort=False)])
rng = np.random.default_rng(SEED)
explanation_rows, explanation_html = [], []
for index, row in explained.iterrows():
    words, delta = word_attributions(row["text_clean"])
    top_words = [word for word, _ in sorted(words, key=lambda pair: pair[1], reverse=True)[:3]]
    explanation_rows.append({"outcome": row["outcome"], "annotator score": row["toxicity"], "top words": top_words,
                             "convergence delta": delta} | deletion_check(words, rng))
    explanation_html.append(f"<p><b>{row['outcome']}</b> (annotator score {row['toxicity']:.2f}, DistilBERT "
                            f"{row['distilbert_probability']:.2f}): {coloured_words(words)}</p>")
display(HTML("".join(explanation_html)))
display(pd.DataFrame(explanation_rows, index=explained.index).round(3))

EXPLAINABILITY_INTERPRETATION_PLACEHOLDER

## 10. Ethics and risk

### 10.1 Identity slices

Civil Comments' Hugging Face release has no identity labels, so the slices below use a word list instead: a comment is
in the "muslim" slice if it contains "muslim", "muslims", "islam" or "islamic". That is a proxy. A comment that
mentions a group is not always about that group ("white house", "black market"), and comments about a group do not
always name it. The question is the one from H2: does a harmless comment get flagged more often just because it
mentions an identity?

The false positive rate (FPR) is the share of non-toxic comments that get flagged. The false negative rate (FNR) is
the share of toxic comments that are missed. The ratio column divides a slice's FPR by the FPR on all test comments;
the success criterion was a ratio of at most 2.

In [ ]:
IDENTITY_TERMS = {
    "women": ["woman", "women", "female", "females", "girl", "girls"],
    "men": ["man", "men", "male", "males", "boy", "boys"],
    "black": ["black", "blacks"],
    "white": ["white", "whites"],
    "muslim": ["muslim", "muslims", "islam", "islamic"],
    "jewish": ["jew", "jews", "jewish", "judaism"],
    "christian": ["christian", "christians", "christianity"],
    "lgbtq": ["gay", "gays", "lesbian", "lesbians", "homosexual", "homosexuals", "transgender", "bisexual", "queer",
              "lgbt", "lgbtq"],
}
MIN_TOXIC_PER_SLICE = 100
FPR_RATIO_LIMIT = 2.0


def mentions_any(texts, terms):
    return texts.str.contains(r"\b(?:" + "|".join(terms) + r")\b", case=False, regex=True)


def slice_error_rates(rows):
    toxic = predictions.loc[rows, "label"] == 1
    rates = {"comments": int(rows.sum()), "toxic comments": int(toxic.sum())}
    for name, column in FLAG_COLUMNS.items():
        flagged = predictions.loc[rows, column] == 1
        rates |= {f"FPR {name}": flagged[~toxic].mean(), f"FNR {name}": 1 - flagged[toxic].mean()}
    return rates


slices = {"all comments": pd.Series(True, index=predictions.index)}
slices |= {group: mentions_any(predictions["text_clean"], terms) for group, terms in IDENTITY_TERMS.items()}
slice_table = pd.DataFrame({group: slice_error_rates(rows) for group, rows in slices.items()}).T
for name in MODELS:
    slice_table[f"FPR ratio {name}"] = slice_table[f"FPR {name}"] / slice_table.loc["all comments", f"FPR {name}"]
slice_table["few toxic comments"] = slice_table["toxic comments"] < MIN_TOXIC_PER_SLICE
display(slice_table.round(3))

In [ ]:
ratios = slice_table.drop("all comments")[[f"FPR ratio {name}" for name in MODELS]]
fig, ax = plt.subplots()
ratios.plot.bar(ax=ax, color=["tab:blue", "tab:red"], rot=0)
ax.axhline(1, color="grey", linewidth=0.8)
ax.axhline(FPR_RATIO_LIMIT, color="black", linestyle="--", label=f"success limit ({FPR_RATIO_LIMIT:g}x)")
ax.set_ylabel("FPR in slice / FPR overall")
ax.set_title("How much more often harmless comments are flagged when they mention a group")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "identity_fpr_ratio.png", dpi=150)
plt.show()

### 10.2 Template test

Real comments that mention a group differ in many ways at once. Templates remove that: every sentence below is harmless
by construction and only the identity word changes. Any difference in score comes from the identity word alone (Dixon
et al., 2018, used the same idea).

In [ ]:
TEMPLATES = ["I am a {} person.", "My neighbour is {}.", "Many {} people live on my street.",
             "I had lunch with a {} friend today."]
IDENTITY_WORDS = ["straight", "gay", "lesbian", "transgender", "christian", "muslim", "jewish", "atheist", "white",
                  "black", "asian", "american", "young", "old"]

template_sentences = pd.DataFrame([{"identity": word, "sentence": template.format(word)}
                                   for word in IDENTITY_WORDS for template in TEMPLATES])
template_sentences["TF-IDF + LR"] = toxic_proba(baseline, template_sentences["sentence"])
template_sentences["DistilBERT"] = toxic_probabilities(predict_logits(template_sentences["sentence"].tolist()))
template_scores = template_sentences.groupby("identity", sort=False)[MODELS].mean().sort_values("DistilBERT")

fig, ax = plt.subplots()
template_scores.plot.barh(ax=ax, color=["tab:blue", "tab:red"])
for name, colour in zip(MODELS, ["tab:blue", "tab:red"]):
    ax.axvline(thresholds[name], color=colour, linestyle="--", linewidth=1, label=f"{name} threshold")
ax.set_xlabel("mean toxic score over four harmless templates")
ax.set_title("Same sentence, different identity word")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "template_test.png", dpi=150)
plt.show()
flagged_templates = {name: int((template_sentences[name] >= thresholds[name]).sum()) for name in MODELS}
print(f"Harmless template sentences flagged: {flagged_templates} out of {len(template_sentences)}")

### 10.3 Confidence calibration

A moderator who sees "0.9" will read it as "90% likely to be toxic". The reliability diagram checks whether that reading
is fair: comments are grouped into ten bins by predicted score, and each bin's average score is plotted against the
share of its comments that really are toxic. A calibrated model sits on the diagonal. The expected calibration error
(ECE) is the average gap, weighted by how many comments fall in each bin.

Temperature scaling (Guo et al., 2017) is one possible fix. It divides DistilBERT's logits by a single number fitted on
the validation set. The ranking of comments, and with it the PR-AUC, does not change.

In [ ]:
CALIBRATION_BINS = 10


def expected_calibration_error(labels, probabilities, bins=CALIBRATION_BINS):
    labels, probabilities = np.asarray(labels), np.asarray(probabilities)
    bin_ids = np.digitize(probabilities, np.linspace(0, 1, bins + 1)[1:-1])
    return float(sum(np.mean(bin_ids == b) * abs(labels[bin_ids == b].mean() - probabilities[bin_ids == b].mean())
                     for b in np.unique(bin_ids)))


def fit_temperature(logits, labels):
    return minimize_scalar(lambda temperature: log_loss(labels, toxic_probabilities(logits, temperature)),
                           bounds=(0.05, 10), method="bounded").x


temperature = fit_temperature(validation_logits, validation["label"])
calibrated = {"TF-IDF + LR": test_probabilities["TF-IDF + LR"], "DistilBERT": test_probabilities["DistilBERT"],
              f"DistilBERT, temperature {temperature:.2f}": toxic_probabilities(test_logits, temperature)}

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], color="grey", linestyle=":", label="perfect calibration")
for (name, probabilities), colour in zip(calibrated.items(), ["tab:blue", "tab:red", "tab:green"]):
    observed, predicted = calibration_curve(test["label"], probabilities, n_bins=CALIBRATION_BINS)
    ece = expected_calibration_error(test["label"], probabilities)
    ax.plot(predicted, observed, marker="o", color=colour, label=f"{name} (ECE {ece:.3f})")
ax.set_xlabel("mean predicted toxic score in bin")
ax.set_ylabel("share of comments in bin that are toxic")
ax.set_title("Reliability diagram, test set")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "reliability_diagram.png", dpi=150)
plt.show()

In [ ]:
results |= {
    "identity slices": slice_table.to_dict(orient="index"),
    "template test mean scores": template_scores.to_dict(orient="index"),
    "harmless template sentences flagged": flagged_templates,
    "expected calibration error": {name: expected_calibration_error(test["label"], probabilities)
                                   for name, probabilities in calibrated.items()},
    "temperature": temperature,
    "error categories": error_counts.to_dict(orient="index"),
    "training": {"runtime minutes": training_log[-1]["train_runtime"] / 60, "best step": int(best_step),
                 "best validation PR-AUC": validation_curve.max()},
}
RESULTS_PATH.write_text(json.dumps(results, indent=2, default=float))
print(f"Saved {RESULTS_PATH}")

CALIBRATION_INTERPRETATION_PLACEHOLDER

### 10.4 Misuse scenarios and affected stakeholders

| Scenario | Who is hurt | How |
|---|---|---|
| The score removes comments automatically instead of ordering a queue | people who write about their own identity | harmless comments that mention "gay", "muslim" or "black" are flagged more often (10.1, 10.2), so the groups a moderation policy is meant to protect lose their comments first |
| Dialect and reclaimed words | speakers of African American English; LGBTQ commenters who use reclaimed slurs among themselves | Sap et al. (2019) found that crowd annotators rate African American English as more offensive, and a model trained on those labels learns the same bias. This data has no dialect labels, so the notebook cannot measure it |
| Moderators over-trust the score | moderators, and through them commenters | a number looks objective. Under time pressure a moderator may approve whatever scores high instead of reading it (automation bias) |
| Deliberate evasion | readers of the comment section | trolls who misspell words or space them out ("i d i o t") can push a comment below the threshold. Word pieces soften this for DistilBERT but do not remove it |
| Reuse somewhere else | users of other platforms, languages or later years | the model learned North American news comments from 2015 to 2017. Slang, topics and community norms differ elsewhere, and none of the numbers in this notebook transfer |
| Profiling users | commenters with unpopular views | averaged per user, the score could be used to rank and silence people instead of comments |

### 10.5 Mitigations

1. **Order the queue, never remove.** The model only decides what a moderator reads first. Every removal stays a human
   decision and is logged together with the score, so an audit can spot moderators who simply follow the number.
2. **Set the threshold from a precision target.** Moderators and the site agree on how many flagged comments must
   really be toxic, and the threshold is derived from a freshly labelled sample each quarter, because topics drift.
3. **Gate every model update on sections 10.1 and 10.2.** A new model is not deployed if any identity slice has an FPR
   ratio above 2, or if any harmless template sentence crosses the threshold.
4. **Fix the training data.** Counterfactual augmentation copies harmless training comments that mention a group and
   swaps the identity word ("gay" for "straight", "muslim" for "christian"), so identity words also appear in harmless
   contexts. Dixon et al. (2018) added non-toxic identity examples for the same reason.
5. **Show calibrated scores with a caveat.** Moderators see the temperature-scaled score and the top attributed words,
   labelled as a hint, not a verdict.
6. **Offer appeals.** Commenters can contest a removal, and overturned decisions become new labelled training data.

## 11. Final recommendation

RECOMMENDATION_PLACEHOLDER